# Introduction to Artificial Intelligence - Homework Assignment 09 (5 pts.)
- Name:
- NETID:

**Following the Gradient Through Time**
Due December 2, 2026.

This assignment covers:

- Backward passes for each layer of HW08's network, including a linear layer on 3-D input
- Backpropagation through time (BPTT): sending the gradient back through the recurrent block
- Softmax, a cross-entropy loss that ignores padding, its gradient, and accuracy
- Training the scratch network with gradient descent, and the same network in PyTorch
- Decoding the held-out phrases and the evidence note

| Task | Description | Points |
|---|---|---:|
| 00 | Reopen the cipher evidence | 0 |
| 01 | Backpropagate through the layers | 2.00 |
| 02 | The loss, its gradient, and the training loop; explain the average | 1.375 |
| 03 | Train the scratch and PyTorch networks | 1.00 |
| 04 | Decode the evidence; compare the two accuracies | 0.125 |
| 05 | Report the results to Caulfield | 0.50 |
| 06 | Export the notebook | 0 |
| | **Total** | **5.00** |

Run the cells in order and answer the written questions. This assignment trains a small recurrent network; each training cell takes under a minute and prints its progress as it goes.

### *Story Progression*

Last week's network did exactly what an untrained network should: it read the encoded note and produced confident nonsense. Caulfield had it pinned to the board next to the original, which the intern thought was a little unkind.

The department has supplied 900 training pairs of encoded and plain phrases, plus separate validation and test sets, all written with the same substitution as the note. The intern points out that once you know the trick (every letter is shifted 13 places), you could decode the note with a pencil. Caulfield agrees, and says that's exactly why it's a good test. If a decoder we train ourselves reproduces a code we can check by hand, we'll know how far to trust it when the next piece of evidence isn't so simple.

This week we teach the recurrent network from HW08. The recipe is HW07's: a loss, its gradient, and many small steps downhill. What's new is that the same weights are used at **every position** in the sequence, so the gradient has to be traced backward through time.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/lab04/rot13_confirmed.png" class="img-responsive" alt="Evidence 1: The department's paired phrases: the same code, with the answers written out." style="display: block; max-width: 640px; width: 100%; height: auto; margin: 0 auto;"/>
    <figcaption style="text-align: center"><b>Evidence 1: The department's paired phrases: the same code, with the answers written out.</b></figcaption>
</div>

## Task 00: Reopen the Cipher Evidence

### Task 00-1: Description (0 pts.)

#### The Same Data as HW08

This cell repeats HW08's setup: the phrase pairs, the note, and the `Rot13Dataset` that owns the 41-token vocabulary, pads each batch, and returns **one-hot** targets `(B, T, 41)`. `device` picks a GPU when one is available.

| File | Contents | When to use it |
|---|---|---|
| `rot13_pairs.json` | Aligned cipher/plain phrase pairs in `train`, `validation`, `test` splits | Training and evaluation |
| `cipher_note.txt` | The forensics lab's checked transcription of the note, still encoded | Task 04 |

The note is never used for training. Nothing to implement here: run the cell and check the printed summary.

### Task 00-1: Code (0 pts.)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/wtheisen/nd-cse-30124-homeworks.git'

try:
    import google.colab
except ImportError:
    # Reuse a local checkout, including when launched from a homework subfolder.
    DATA_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / '.git').exists() and (p / 'evidence').is_dir()), None)
    if DATA_ROOT is None:
        DATA_ROOT = Path.cwd() / 'nd-cse-30124-homeworks'
else:
    # Leave the old checkout before deleting it, including on a setup-cell rerun.
    os.chdir('/content')
    DATA_ROOT = Path('/content/nd-cse-30124-homeworks')
    # Fetch fresh evidence every time this cell runs in Colab.
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)

if not DATA_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(DATA_ROOT)], check=True)

if not (DATA_ROOT / '.git').exists():
    raise RuntimeError(f'{DATA_ROOT} exists but is not a Git checkout. Choose a different clone location.')

# All later relative paths now start at the homework repository root.
os.chdir(DATA_ROOT)
DATA_ROOT = Path.cwd()

import codecs
import json
import math
import string
from collections import namedtuple
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

torch.set_num_threads(2)
torch.manual_seed(42)

EVIDENCE_DIR = DATA_ROOT / 'evidence' / 'homework09'
if any(not (EVIDENCE_DIR / name).is_file() for name in ['rot13_pairs.json', 'cipher_note.txt']):
    raise FileNotFoundError('HW09 evidence is missing. Rerun the setup cell with an internet connection, or check that your local repository contains the evidence/homework09 files.')
os.chdir(EVIDENCE_DIR)

# Supplied from HW08: the phrase pairs, the note, and the dataset that owns the vocabulary.
pairs = json.loads(Path('rot13_pairs.json').read_text())
letter_text = Path('cipher_note.txt').read_text().strip()

device = torch.device('cuda' if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

class Rot13Dataset(Dataset):
    def __init__(self, records, device="cpu", one_hot_targets=True):
        # ---- vocab (owned by the dataset) ----
        self.vocab = ('<PAD>', '<EOS>', '<UNK>', '<SOS>', ' ') + tuple(string.ascii_lowercase + string.digits)  # 41
        self.vocab_size = len(self.vocab)
        self.char2idx = {ch: i for i, ch in enumerate(self.vocab)}
        self.idx2char = {i: ch for i, ch in enumerate(self.vocab)}
        self.padding_idx = self.char2idx['<PAD>']
        self.eos_idx = self.char2idx['<EOS>']

        self.device = device
        self.one_hot_targets = one_hot_targets

        # The cipher unit's phrase pairs: (disguised IDs, plain IDs)
        self.data = [(self.encode(r['cipher']), self.encode(r['plain'])) for r in records]

    # ---------- helpers ----------
    def encode(self, s):  # string -> 1D LongTensor, ending in <EOS>
        return torch.tensor([self.char2idx.get(c, self.char2idx['<UNK>']) for c in s.lower()] + [self.eos_idx], dtype=torch.long)

    def decode(self, idxs):  # 1D LongTensor -> string: stop at <EOS>, skip <PAD>, '?' for other special tokens
        text = []
        for i in idxs:
            i = int(i)
            if i == self.eos_idx:
                break
            if i == self.padding_idx:
                continue
            ch = self.idx2char[i]
            text.append(ch if len(ch) == 1 else '?')
        return ''.join(text)

    # ---------- Dataset API ----------
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]  # (T,), (T,)

    # ---------- Collate (pad, one-hot targets, move to device) ----------
    def collate(self, batch):
        X_list, Y_list = zip(*batch)  # tuples of 1D tensors
        X = pad_sequence(X_list, batch_first=True, padding_value=self.padding_idx)        # (B,T)
        Y_idx = pad_sequence(Y_list, batch_first=True, padding_value=self.padding_idx)    # (B,T)
        if not self.one_hot_targets:
            return X.to(self.device), Y_idx.to(self.device)

        Y = F.one_hot(Y_idx.clamp_min(0), num_classes=self.vocab_size).float()        # (B,T,V)
        return X.to(self.device), Y.to(self.device)

datasets = {name: Rot13Dataset(pairs[name], device=device) for name in ('train', 'validation', 'test')}
dataset = datasets['train']
train_loader = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=dataset.collate,
                          generator=torch.Generator().manual_seed(42))

print('Train / validation / test:', len(datasets['train']), len(datasets['validation']), len(datasets['test']))
print('Vocabulary size:', dataset.vocab_size, '| training batches per epoch:', len(train_loader))
print('Note (first 40 characters):', letter_text[:40])

### Task 00-1: Expected Output (0 pts.)

```text
Train / validation / test: 900 150 150
Vocabulary size: 41 | training batches per epoch: 15
Note (first 40 characters): v ybirq zheqrevat ze gurvfrasyblq ng gur
```

## Task 01: Backpropagate Through the Layers

### Task 01-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/plates/plate_01-1_linear_3d.png" alt="A linear layer on sequences: flatten dY and X across batch and position, then apply HW07's dW, db, dX formulas; the embedding layer inherits the same backward." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Your HW08 `LinearLayer` is in the cell below with its forward pass. Its HW07 `backward` assumed 2-D input `(B, D)`; the network feeds it 3-D input `(B, T, D)`. Complete:

1. `backward(dY)`: flatten `dY` and the cached `self.X` across the batch and sequence dimensions, compute `self.dW` and `self.db`, and return `dX` with the input's 3-D shape.
2. `update(lr)`: one gradient-descent step on `W` and `b`.

`EmbeddingLayer` (supplied from HW08, at the bottom of the cell) inherits both methods: its cached input is the one-hot `(B, T, V)`, so the same code gives its weight gradient.

#### Python Help

```python
dY = torch.zeros(2, 6, 3)
dY.reshape(-1, dY.shape[-1]).shape   # torch.Size([12, 3]): every (phrase, position) is a row
dY.sum(dim=(0, 1)).shape              # torch.Size([3])
```

### Task 01-1: Code (0.5 pts.)

In [ ]:
class LinearLayer:
    """
    A fully connected (dense) layer that performs a linear transformation.

    Attributes:
        W (torch.Tensor): Weight matrix with shape (output_dim, input_dim).
        b (torch.Tensor): Bias vector with shape (output_dim,).
        X (torch.Tensor): Cached input used during the forward pass.
        dW (torch.Tensor): Gradient with respect to the weights.
        db (torch.Tensor): Gradient with respect to the biases.
    """
    def __init__(self, input_dim, output_dim, device='cpu'):
        self.device = device
        self.W = torch.randn(output_dim, input_dim, device=self.device) * math.sqrt(2.0 / input_dim)
        self.b = torch.zeros(output_dim, device=self.device)

    def forward(self, X):
        self.X = X
        return X @ self.W.T + self.b

    def backward(self, dY):
        """
        Compute the backward pass of the linear layer.

        Args:
            dY (torch.Tensor): Gradient with respect to the output of the linear layer (batch_size, sequence_length, output_dim)

        Returns:
            dX (torch.Tensor): Gradient with respect to the input of the linear layer (batch_size, sequence_length, input_dim)

        Notes:
            This is trickier than in Homework 07 because our input is now 3D (batch_size, sequence_length, input_dim)
            To calculate dW we use a similar idea to im2col and flatten the input matrix across the batch_size and sequence_length dimensions.
            We then do the same thing for X and then can use a simple 2D matmul to calculate dW directly.
        """
        # TODO: Flatten dY and X across the batch and sequence dimensions, then compute dW, db, and dX
        pass

    def update(self, lr):
        """
        Update the parameters of the layer using gradient descent.

        Args:
            lr (float): Learning rate for the parameter update.
        """
        # TODO: Update the weights and biases of the layer using the learning rate
        pass

# Supplied from HW08: the embedding layer, the tanh forward pass, and the network's layers.
class EmbeddingLayer(LinearLayer):
    def __init__(self, vocab_size, embed_dim, device="cpu"):
        super().__init__(input_dim=vocab_size, output_dim=embed_dim, device=device)
        self.vocab_size = vocab_size

    def forward(self, X_idx):
        # Build one-hot so LinearLayer can do its usual math (backward is inherited from LinearLayer)
        self.X = F.one_hot(X_idx, num_classes=self.vocab_size).float()  # (B,T,V)
        return self.X @ self.W.T  # (B,T,V) @ (V,E) -> (B,T,E)

# Supplied check: one tiny 3-D layer against autograd.
torch.manual_seed(1)
toy = LinearLayer(4, 3, device=device)
X = torch.randn(2, 5, 4, device=device, requires_grad=True)
W = toy.W.clone().requires_grad_(True); b = toy.b.clone().requires_grad_(True)
dY = torch.randn(2, 5, 3, device=device)
(X @ W.T + b).backward(dY)
toy.forward(X.detach())
dX = toy.backward(dY)
print('dW', tuple(toy.dW.shape), 'matches autograd:', torch.allclose(toy.dW, W.grad, atol=1e-5))
print('db', tuple(toy.db.shape), 'matches autograd:', torch.allclose(toy.db, b.grad, atol=1e-5))
print('dX', tuple(dX.shape), 'matches autograd:', torch.allclose(dX, X.grad, atol=1e-5))

### Task 01-1: Expected Output (0 pts.)

```text
dW (3, 4) matches autograd: True
db (3,) matches autograd: True
dX (2, 5, 4) matches autograd: True
```

### Task 01-2: Description (0 pts.)

#### Your Task

Complete `Tanh.backward` and `Tanh.update`. The slope of tanh can be written using its **output**: if $h = \tanh(a)$, then $\frac{dh}{da} = 1 - h^2$. `RecurrentBlock.backward` (next task) passes each step's cached `h_at_timestep` in and multiplies the returned slope by the incoming gradient itself. tanh has no parameters, so `update` does nothing.

#### Python Help

```python
h = torch.tanh(torch.tensor([0., 1.]))   # tensor([0.0000, 0.7616])
1 - h**2                                  # tensor([1.0000, 0.4200])
```

### Task 01-2: Code (0.25 pts.)

In [ ]:
class Tanh:
    """
    The tanh activation function

    Attributes:
       None
    """
    def forward(self, X):
        # Supplied from HW08
        return torch.tanh(X)

    def backward(self, dA):
        """
        Compute the backward pass of the tanh activation function

        Args:
            dA (torch.Tensor): The tanh OUTPUT h at this timestep, shape (batch_size, hidden_size)
                               (RecurrentBlock.backward passes h_at_timestep in)

        Returns:
            torch.Tensor: the derivative of tanh at that point, 1 - h^2
        """
        # TODO: Return the derivative of tanh
        pass

    def update(self, lr):
        """
        Update the parameters of the layer using gradient descent.

        Args:
            lr (float): Learning rate for the parameter update.
        """
        # TODO: Update the parameters with learning rate lr
        pass

h = Tanh().forward(torch.tensor([-2., 0., 0.5, 3.], device=device))
print('Slope 1 - h^2 at a = -2, 0, 0.5, 3:', [round(v, 4) for v in Tanh().backward(h).tolist()])

### Task 01-2: Expected Output (0 pts.)

```text
Slope 1 - h^2 at a = -2, 0, 0.5, 3: [0.0707, 1.0, 0.7864, 0.0099]
```

### Task 01-3: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/plates/plate_01-3a_two_jobs.png" alt="Unrolling the recurrent block: each hidden state feeds this position's output and the next step, so its gradient is the sum of both paths, and the shared W, U, b gradients accumulate over positions." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/plates/plate_01-3b_one_step_backward.png" alt="Table of one backward step at position t: dcurrent, the tanh slope, dz, the dW, dU, db contributions, dX, and dh_next, with the shapes multiplied." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/plates/plate_01-3c_one_number_rnn.png" alt="Worked example of BPTT on a hidden-size-1 RNN over two steps, showing the future path changing the sign of the first step's gradient." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete `RecurrentBlock.backward(d_outputs)` and `update`. The forward pass is HW08's. Walk the timesteps **backward**; at each one:

1. `dcurrent`: this position's gradient from the output layer, plus `dh_next`, the gradient arriving from the future.
2. Read this step's `Grad_Info` (index `backwards_timestep + 1`: entry 0 is the zero start) and multiply `dcurrent` by the tanh slope to get `dz_at_timestep`.
3. **Add** this step's contributions to `dW`, `dU`, and `db`; store this step's `dX`; and set `dh_next` for the step before.

`update(lr)` takes one gradient-descent step on `W`, `U`, and `b`. The supplied check compares every gradient with autograd.

#### Python Help

```python
for t in reversed(range(4)):   # 3, 2, 1, 0
    ...
dz.T @ x                       # (hidden, B) @ (B, input): one step's contribution to dW
```

### Task 01-3: Code (1.25 pts.)

In [ ]:
class RecurrentBlock:
    """
    A Recurrent Block

    Attributes:
        W (torch.Tensor): Input Weight matrix with shape (hidden_size, input_dim).
        U (torch.Tensor): Hidden Weight matrix with shape (hidden_size, hidden_size).
        b (torch.Tensor): Bias vector with shape (hidden_size).
        Grad_Info (namedtuple): Cached hidden state info using the Grad_Info tuple made in forward and used in backward
        dW (torch.Tensor): Gradient with respect to the input weights.
        dU (torch.Tensor): Gradient with respect to the hidden weights.
        db (torch.Tensor): Gradient with respect to the biases.
    """

    Grad_Info = namedtuple('Grad_Info', ['x_at_timestep', 'h_at_timestep', 'h_prev_at_timestep'])

    def __init__(self, input_dim, hidden_size, device='cpu'):
        self.input_dim = input_dim      # This should match the embedding size.
        self.hidden_size = hidden_size
        self.device = device

        self.W = torch.randn(hidden_size, input_dim, device=self.device) * math.sqrt(2.0 / input_dim)
        self.U = torch.randn(hidden_size, hidden_size, device=self.device) * math.sqrt(2.0 / input_dim)
        self.b = torch.zeros(hidden_size, device=self.device)

        self.activation = Tanh()
        self.hidden_states = []

    def forward(self, X):
        # Supplied from HW08
        batch_size, seq_len, _ = X.shape

        outputs = []
        self.hidden_states = [self.Grad_Info(
            x_at_timestep=None,
            h_at_timestep=torch.zeros((batch_size, self.hidden_size), device=self.device),
            h_prev_at_timestep=None
        )]

        for timestep in range(seq_len):
            x_at_timestep = X[:, timestep, :]

            h_prev_at_timestep = self.hidden_states[-1].h_at_timestep
            z_at_timestep = x_at_timestep @ self.W.t() + h_prev_at_timestep @ self.U.t() + self.b
            h_at_timestep = self.activation.forward(z_at_timestep)

            self.hidden_states.append(self.Grad_Info(
                x_at_timestep = x_at_timestep,
                h_at_timestep = h_at_timestep,
                h_prev_at_timestep = h_prev_at_timestep))

            outputs.append(h_at_timestep)

        # Stack outputs along the time-axis: (batch_size, seq_len, hidden_size)
        self.outputs = torch.stack(outputs, dim=1)
        return self.outputs

    def backward(self, d_outputs):
        """
        Compute the backward pass of the recurrent block using backpropagation through time (BPTT)

        Args:
            d_outputs (torch.Tensor): Gradient with respect to outputs with shape: (batch_size, sequence_len, hidden_size)

        Returns:
            torch.Tensor: dX after BPTT is performed with shape: (batch_size, sequence_len, embedding_size)
        """
        # d_outputs:
        batch_size, seq_len, hidden_size = d_outputs.shape

        dW = torch.zeros_like(self.W, device=self.device)
        dU = torch.zeros_like(self.U, device=self.device)
        db = torch.zeros_like(self.b, device=self.device)
        dX = torch.zeros((batch_size, seq_len, self.input_dim), device=self.device)

        dh_next = torch.zeros((batch_size, hidden_size), device=self.device)  # Gradient propagated from future time steps.

        for backwards_timestep in list(reversed(range(seq_len))):
            # TODO: Compute the backward pass for each item in the sequence
            pass

        self.dW = dW
        self.dU = dU
        self.db = db

        return dX

    def update(self, lr):
        """
        Update the parameters of the block using gradient descent.

        Args:
            lr (float): Learning rate for the parameter update.
        """
        # TODO: Update the parameters with learning rate lr
        pass

# Supplied check: a small block against autograd, with and without the time path.
torch.manual_seed(2)
block = RecurrentBlock(3, 4, device=device)
X = torch.randn(2, 5, 3, device=device)
d_out = torch.randn(2, 5, 4, device=device)
W, U, b = (p.clone().requires_grad_(True) for p in (block.W, block.U, block.b))
Xg = X.clone().requires_grad_(True)
h, states = torch.zeros(2, 4, device=device), []
for t in range(5):
    h = torch.tanh(Xg[:, t] @ W.T + h @ U.T + b)
    states.append(h)
(torch.stack(states, dim=1) * d_out).sum().backward()
block.forward(X)
dX = block.backward(d_out)
for name, mine, ref in [('dW', block.dW, W.grad), ('dU', block.dU, U.grad), ('db', block.db, b.grad), ('dX', dX, Xg.grad)]:
    print(f'{name} {str(tuple(mine.shape)):8} matches autograd: {torch.allclose(mine, ref, atol=1e-5)}')

### Task 01-3: Expected Output (0 pts.)

```text
dW (4, 3)   matches autograd: True
dU (4, 4)   matches autograd: True
db (4,)     matches autograd: True
dX (2, 5, 3) matches autograd: True
```

## Task 02: The Loss, Its Gradient, and the Training Loop

### Task 02-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/plates/plate_02-1a_masked_loss.png" alt="Cross-entropy averaged over real positions only: per-position loss from one-hot targets, a mask from the padding ID, divided by the number of real positions." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/plates/plate_02-1b_gradient_at_top.png" alt="The logit gradient is (probabilities minus one-hot target) divided by N at real positions and zero at padding, derived from the softmax." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/plates/plate_02-1c_worked_example.png" alt="Worked example on a three-position toy phrase: probabilities, one-hot targets, and the resulting gradient with the padded position zeroed." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/plates/plate_02-1d_training_loop.png" alt="The training loop: forward as probabilities, loss and accuracy, backward, then update every layer, with learning rate 0.5." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

The network's `__init__` and `forward` are HW08's. Complete:

1. `softmax(X)`: probabilities over the last dimension. Subtract the row maximum before `exp` (as in HW06).
2. `cross_entropy(Y_hat, Y)`: the mean of $-\log$ (probability of the target) over **real** positions only. `Y` is one-hot, so `-(Y * log_probs).sum(dim=-1)` picks out the target's term.
3. `get_accuracy(Y_hat, Y)`: the fraction of real positions where the most likely token is the target.
4. `backward(Y_hat, Y)`: the gradient at the logits is `(Y_hat - Y)`, zeroed at padding and divided by the number of real positions. Pass it backward through the layers in reverse order.
5. In `train`, the three marked steps for each batch: the forward pass **as probabilities** (`eval=True`), the backward pass, and an update for every layer.

The supplied check runs a two-phrase batch through a small network and compares every gradient with autograd.

#### Python Help

```python
torch.amax(X, dim=-1, keepdim=True)   # the largest logit at every position, shape (B, T, 1)
Y.argmax(dim=-1)                      # one-hot back to IDs, shape (B, T)
mask.unsqueeze(-1)                    # (B, T) -> (B, T, 1), so it broadcasts over the vocabulary
```

### Task 02-1: Code (1.25 pts.)

In [ ]:
class RecurrentNeuralNetwork:
    def __init__(self, vocab_size, embed_size, hidden_size, padding_idx=0, device='cpu'):
        # Supplied from HW08
        self.vocab_size = vocab_size
        self.embed_size = embed_size
        self.hidden_size = hidden_size
        self.padding_idx = padding_idx
        self.device = device

        embedding_layer = EmbeddingLayer(vocab_size, embed_size, device=device)
        recurrent_block = RecurrentBlock(embed_size, hidden_size, device=device)
        output_layer = LinearLayer(hidden_size, vocab_size, device=device)
        self.layers = [embedding_layer, recurrent_block, output_layer]

    def forward(self, X, eval=False):
        """
        Args:
          X: (batch_size, seq_len) with integer token indices.
        Returns:
          logits (eval=False) or softmax probabilities (eval=True): (batch_size, seq_len, vocab_size)
        """
        # Supplied from HW08
        for layer in self.layers:
            X = layer.forward(X)

        return X if not eval else self.softmax(X)

    def backward(self, Y_hat, Y):
        """
        Args:
          Y_hat: softmax probabilities, (batch_size, seq_len, vocab_size)
          Y: one-hot targets, (batch_size, seq_len, vocab_size)
        """
        # TODO: Calculate the gradient of the loss with respect to the input
        pass

    def softmax(self, X):
        """
        Args:
            X (torch.Tensor): Logits with shape (batch_size, seq_len, vocab_size).

        Returns:
            torch.Tensor: Softmax probabilities over the last dimension, same shape.
        """
        # TODO: Calculate the output of the softmax layer
        pass

    def cross_entropy(self, Y_hat, Y):
        """
        Compute the cross-entropy loss.

        Args:
            Y_hat (torch.Tensor): Predicted probabilities, shape (batch_size, seq_len, vocab_size).
            Y (torch.Tensor): One-hot encoded true labels, same shape.

        Returns:
            torch.Tensor: The average cross-entropy loss over the real (non-PAD) target positions.

        Notes:
            Y_hat is clamped away from 0 to avoid computing log(0).
        """

        # TODO: Calculate the cross-entropy loss
        pass

    def get_accuracy(self, Y_hat, Y):
        """
        Compute the classification accuracy over the real (non-PAD) target positions.

        Args:
            Y_hat (torch.Tensor): Logits or probabilities, shape (batch_size, seq_len, vocab_size).
            Y (torch.Tensor): One-hot encoded true labels, same shape.

        Returns:
            torch.Tensor: Accuracy as a fraction between 0 and 1.
        """

        # TODO: Calculate the accuracy of the network
        pass

    def test_on_letter(self, dataset):
        x = dataset.encode(letter_text).unsqueeze(0).to(self.device)  # (1, T)

        preds = self.forward(x, eval=True)
        preds = preds.argmax(dim=-1)  # (1, T)
        predicted_deciphered = "".join(dataset.idx2char[int(i)] for i in preds.squeeze(0))

        print(f"\tInput (ROT13): {letter_text}")
        print(f"\tPredicted original: {predicted_deciphered}")

        return predicted_deciphered

    def train(self, train_loader, epochs=100, learning_rate=0.5, verbose=True):
        """
        Train the neural network using mini-batch gradient descent.

        Args:
            train_loader (DataLoader): Batches of (X, Y): token IDs (B, T) and one-hot targets (B, T, V).
            epochs (int): Number of epochs for training.
            learning_rate (float): Learning rate for the parameter updates.
            verbose (bool, optional): If True, prints training progress four times. Default is True.

        Returns:
            dict: A dictionary containing:
                - 'loss_history': List of loss values for each epoch.
                - 'accuracy_history': List of accuracy values for each epoch.
        """
        loss_history = []
        accuracy_history = []

        for i in range(epochs):
            batch_losses = []
            batch_accuracies = []

            for X_batch, Y_batch in train_loader:
                # Forward propagation
                # TODO: Calculate the output of the network (as softmax probabilities)

                # Calculate metrics for the whole epoch
                loss = self.cross_entropy(Y_hat_batch, Y_batch)
                accuracy = self.get_accuracy(Y_hat_batch, Y_batch)

                batch_losses.append(loss.item())
                batch_accuracies.append(accuracy.item())

                # Backward propagation
                # TODO: Calculate the gradients of the loss with respect to the input

                # Update parameters
                # TODO: Update the weights and biases of the layer using the learning rate

            loss_history.append(np.mean(batch_losses))
            accuracy_history.append(np.mean(batch_accuracies))

            if verbose and i % (epochs // 4) == 0:
                print(f"Epoch {i+1}/{epochs}")
                print(f"loss: {loss_history[-1]:.5f}")
                print(f"accuracy: {accuracy_history[-1]:.5f}")
                print('Output test:')
                self.test_on_letter(train_loader.dataset)
                print("-" * 30)

        return {'loss_history': loss_history, 'accuracy_history': accuracy_history}

# Supplied check: every gradient of a small network against autograd.
torch.manual_seed(3)
small = RecurrentNeuralNetwork(vocab_size=dataset.vocab_size, embed_size=4, hidden_size=5, padding_idx=dataset.padding_idx, device=device)
x_small, y_small = dataset.collate([(dataset.encode('nnn'), dataset.encode('aaa')), (dataset.encode('b'), dataset.encode('o'))])
probs = small.forward(x_small, eval=True)
print(f'Toy batch loss: {small.cross_entropy(probs, y_small).item():.4f} | accuracy: {small.get_accuracy(probs, y_small).item():.2f}')
small.backward(probs, y_small)
emb, rec, out = small.layers
params = {'embedding W': emb.W, 'recurrent W': rec.W, 'recurrent U': rec.U, 'recurrent b': rec.b, 'output W': out.W, 'output b': out.b}
leaves = {name: p.clone().requires_grad_(True) for name, p in params.items()}
h = torch.zeros(2, 5, device=device); logits = []
E = F.one_hot(x_small, dataset.vocab_size).float() @ leaves['embedding W'].T
for t in range(x_small.shape[1]):
    h = torch.tanh(E[:, t] @ leaves['recurrent W'].T + h @ leaves['recurrent U'].T + leaves['recurrent b'])
    logits.append(h @ leaves['output W'].T + leaves['output b'])
mask = (y_small.argmax(dim=-1) != dataset.padding_idx).float()
loss = (-(y_small * F.log_softmax(torch.stack(logits, dim=1), dim=-1)).sum(dim=-1) * mask).sum() / mask.sum()
loss.backward()
mine = {'embedding W': emb.dW, 'recurrent W': rec.dW, 'recurrent U': rec.dU, 'recurrent b': rec.db, 'output W': out.dW, 'output b': out.db}
for name in params:
    print(f'{name:12} matches autograd: {torch.allclose(mine[name], leaves[name].grad, atol=1e-6)}')

### Task 02-1: Expected Output (0 pts.)

The loss is exact; every gradient line must read `True`.

```text
Toy batch loss: 3.8931 | accuracy: 0.00
embedding W  matches autograd: True
recurrent W  matches autograd: True
recurrent U  matches autograd: True
recurrent b  matches autograd: True
output W     matches autograd: True
output b     matches autograd: True
```

### Task 02-2: Written Response (0.125 pts.)

In `cross_entropy` and `backward`, why do we divide by the number of real target positions $N$ (`normalizer`), rather than by the batch size $B$ or by $B \times T$? Give one concrete problem each wrong choice would cause.

* `[ANSWER]`

### *Story Progression*

Two independent methods agree on every gradient, including the embedding column that `'nnn'` used three times. Caulfield is satisfied. The intern is mostly relieved that the version without the time path didn't pass, because that was the version he'd written first.

The intern points out that 262 characters is a long way for a gradient to travel. Caulfield points out that this particular code never needs to look back: each character decodes on its own, so the network won't need long-distance memory to get it right. Other evidence might not be so kind, which is a problem for HW10. For now, we train.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework05/rnn_training.png" class="img-responsive" alt="Evidence 2: Training begins. The intern settles in to watch the validation loss fall." style="display: block; max-width: 640px; width: 100%; height: auto; margin: 0 auto;"/>
    <figcaption style="text-align: center"><b>Evidence 2: Training begins. The intern settles in to watch the validation loss fall.</b></figcaption>
</div>

## Task 03: Train the Networks

### Task 03-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/plates/plate_03-1_what_to_expect.png" alt="What training looks like: 15 batches per epoch for 1,500 updates; the loss starts near 3.7 and training accuracy passes 99% by epoch 26." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Create a `RecurrentNeuralNetwork` with `embed_size=32` and `hidden_size=64` (as in HW08) and train it with `model.train(train_loader)`. The defaults run 100 epochs of plain gradient descent at learning rate 0.5, and print progress four times, each with the network's current reading of the note.

The supplied code plots the training curves and reports the accuracy on the validation phrases, which training never sees.

### Task 03-1: Code (0.25 pts.)

In [ ]:
# TODO: Instantiate and train our RNN

# Supplied: the training curves and the held-out validation accuracy.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(history['loss_history']); axes[0].set_ylabel('training loss (log scale)')
axes[1].plot(history['accuracy_history']); axes[1].set_ylabel('training token accuracy')
for ax in axes:
    ax.set_xlabel('epoch'); ax.grid(alpha=0.3)
fig.suptitle('Scratch RNN: plain gradient descent, learning rate 0.5')
fig.tight_layout()
plt.show()
X_val, Y_val = datasets['validation'].collate(list(datasets['validation']))
print(f"Validation token accuracy: {model.get_accuracy(model.forward(X_val), Y_val).item():.4f}")

### Task 03-1: Reference Output (0 pts.)

Your numbers will differ slightly with the random starting weights. Training accuracy should pass 99% by epoch 26, the note should read correctly from then on, and validation accuracy should be at or very near 1.0000.

```text
Epoch 1/100
loss: 3.14377
accuracy: 0.21167
Output test:
	Input (ROT13): v ybirq zheqrevat ze gurvfrasyblq ng gur rfgngr gur tnf jnf gur cresrpg zheqre jrncba vz fher vyy trg njnl jvgu vg nf jryy ubcrshyyl abobql svtherf bhg gur pbzovangvba bs gur cnqybpx cynl ba ybpxre 69 ba gur frpbaq sybbe bs phfuvat bgurejvfr v nz va erny gebhoyr
	Predicted original: i re e  waroerina wr weeisenlloed wa wen ladaie lal wai wai oen lerlei<EOS> weroee waelon wa lare wol wei waal w de wn wo lell wooe llla n oloo laaare  lat cen woooine aoe oa len ladoei  llel s  woeeer wa oe cen weoled wlooe lo looe na woeeeiele w a  wn wenl aroen ea
------------------------------
Epoch 26/100
loss: 0.05255
accuracy: 0.99922
Output test:
	Input (ROT13): v ybirq zheqrevat ze gurvfrasyblq ng gur rfgngr gur tnf jnf gur cresrpg zheqre jrncba vz fher vyy trg njnl jvgu vg nf jryy ubcrshyyl abobql svtherf bhg gur pbzovangvba bs gur cnqybpx cynl ba ybpxre 69 ba gur frpbaq sybbe bs phfuvat bgurejvfr v nz va erny gebhoyr
	Predicted original: i loved murdering mr theisenfloyd at the estate the gas was the perfect murder weapon im sure ill get away with it as well hopefully nobody figures out the combination of the padlock play on locker 69 on the second floor of cushing otherwise i am in real trouble<EOS>
------------------------------
Epoch 51/100
loss: 0.01433
accuracy: 1.00000
Output test:
	Input (ROT13): v ybirq zheqrevat ze gurvfrasyblq ng gur rfgngr gur tnf jnf gur cresrpg zheqre jrncba vz fher vyy trg njnl jvgu vg nf jryy ubcrshyyl abobql svtherf bhg gur pbzovangvba bs gur cnqybpx cynl ba ybpxre 69 ba gur frpbaq sybbe bs phfuvat bgurejvfr v nz va erny gebhoyr
	Predicted original: i loved murdering mr theisenfloyd at the estate the gas was the perfect murder weapon im sure ill get away with it as well hopefully nobody figures out the combination of the padlock play on locker 69 on the second floor of cushing otherwise i am in real trouble<EOS>
------------------------------
Epoch 76/100
loss: 0.00718
accuracy: 1.00000
Output test:
	Input (ROT13): v ybirq zheqrevat ze gurvfrasyblq ng gur rfgngr gur tnf jnf gur cresrpg zheqre jrncba vz fher vyy trg njnl jvgu vg nf jryy ubcrshyyl abobql svtherf bhg gur pbzovangvba bs gur cnqybpx cynl ba ybpxre 69 ba gur frpbaq sybbe bs phfuvat bgurejvfr v nz va erny gebhoyr
	Predicted original: i loved murdering mr theisenfloyd at the estate the gas was the perfect murder weapon im sure ill get away with it as well hopefully nobody figures out the combination of the padlock play on locker 69 on the second floor of cushing otherwise i am in real trouble<EOS>
------------------------------
Validation token accuracy: 1.0000
```

### Task 03-2: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/plates/plate_03-2_library_loop.png" alt="The same training loop in PyTorch: zero_grad, forward, flatten, CrossEntropyLoss with ignore_index, backward, Adam step." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

`CharRNN` is HW08's. Complete the three marked parts of the training cell:

1. Create `rnn_model`, an `Adam` optimizer `rnn_optimizer` with learning rate 0.001, and a loss `rnn_loss_fn`: `nn.CrossEntropyLoss` with `ignore_index` set to the padding ID, so padding is never scored.
2. The forward pass: `logits` for the batch.
3. The loss on the flattened logits and targets, then `backward()` and an optimizer step.

The supplied loop clears the gradients, flattens the batch, and reports the loss, the accuracy on real characters, and the note's reading every 25 epochs.

#### Useful Links

* [`nn.CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) · [`torch.optim.Adam`](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html)

### Task 03-2: Code (0.75 pts.)

In [ ]:
# Supplied from HW08
vocab_size = dataset.vocab_size
embed_size = 32
hidden_size = 64

class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(input_size=embed_size, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h0=None):
        x = self.embed(x)              # -> (batch, seq_len, embed_size) float32
        out, _ = self.rnn(x, h0)       # -> (batch, seq_len, hidden_size)
        logits = self.fc(out)          # -> (batch, seq_len, vocab_size)
        return logits

torch.manual_seed(42)
# TODO: Instantiate the model, optimizer, and loss function

rnn_model.train()
for epoch in range(100):
    total_loss = 0.0
    correct = 0
    count = 0

    for xb, yb in train_loader:
        rnn_optimizer.zero_grad()

        # TODO: Call the forward pass

        B, T, V = logits.shape

        # If yb is one-hot (B, T, V), convert to indices (B, T)
        if yb.dim() == 3:
            # Assumes last dim is vocab dimension
            yb = yb.argmax(dim=-1)          # (B, T)

        # flatten for CrossEntropyLoss
        logits_flat = logits.reshape(B * T, V)  # (B*T, vocab)
        yb_flat = yb.reshape(B * T)             # (B*T,)

        # TODO: Calculate the loss and backpropagate

        total_loss += loss.item()

        preds = logits_flat.argmax(dim=-1)
        real = yb_flat != dataset.padding_idx          # count real characters only, as get_accuracy does
        correct += (preds[real] == yb_flat[real]).sum().item()
        count += real.sum().item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / count

    if epoch % 25 == 0:
        print(f"Epoch {epoch+1} | loss: {avg_loss:.5f} | accuracy: {accuracy:.5f}")
        rnn_model.eval()
        with torch.no_grad():
            x = dataset.encode(letter_text).unsqueeze(0).to(device)  # (1, T)
            preds = rnn_model(x).argmax(dim=-1)  # (1, T)
        predicted_deciphered = "".join(dataset.idx2char[int(i)] for i in preds.squeeze(0))
        print(f"\tPredicted original: {predicted_deciphered}")
        rnn_model.train()

### Task 03-2: Reference Output (0 pts.)

Your numbers will differ slightly. Accuracy should reach 1.0000 within the first few dozen epochs and the note should read correctly.

```text
Epoch 1 | loss: 3.31550 | accuracy: 0.26080
	Predicted original: i  oued whnderln   r   el enclowv a    e e  a e   e  a  wa    e  ercec   hnder wen on lo  hre ll   e  alnm li   l  a  we     uecul   lo4od  ci ure  ou    e  om<EOS>una eol oc   e  ad ock  la  ol  ocker e  ol   e  ecold cloor  c  u  ln     erwu e l a  ln ren   roulue 
Epoch 26 | loss: 0.01978 | accuracy: 1.00000
	Predicted original: i loved murdering mr theisenfloyd at the estate the gas was the perfect murder weapon im sure ill get away with it as well hopefully nobody figures out the combination of the padlock play on locker 69 on the second floor of cushing otherwise i am in real trouble<EOS>
Epoch 51 | loss: 0.00509 | accuracy: 1.00000
	Predicted original: i loved murdering mr theisenfloyd at the estate the gas was the perfect murder weapon im sure ill get away with it as well hopefully nobody figures out the combination of the padlock play on locker 69 on the second floor of cushing otherwise i am in real trouble<EOS>
Epoch 76 | loss: 0.00238 | accuracy: 1.00000
	Predicted original: i loved murdering mr theisenfloyd at the estate the gas was the perfect murder weapon im sure ill get away with it as well hopefully nobody figures out the combination of the padlock play on locker 69 on the second floor of cushing otherwise i am in real trouble<EOS>
```

## Task 04: Decode the Evidence

### Task 04-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/plates/plate_04-1a_two_accuracies.png" alt="Token accuracy versus exact-phrase accuracy: one wrong character in 30 is 97% by tokens but a complete miss by phrases. Evaluate on the test set once." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/plates/plate_04-1b_how_the_network_reads.png" alt="Greedy decoding: encode the note, one forward pass, argmax at every position, stop at the first EOS; checked against codecs ROT13." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

There's no code to write here. The supplied cell scores both trained networks on the 150 held-out test phrases, lists the scratch network's mistakes, and reads the note.

### Task 04-1: Code (0 pts.)

In [ ]:
# Supplied: both trained networks on the held-out test phrases and the note.
test = datasets['test']
X_test, Y_test = test.collate(list(test))
Y_idx = Y_test.argmax(dim=-1)
real = Y_idx != test.padding_idx
rnn_model.eval()
with torch.no_grad():
    readings = {'Scratch RNN': model.forward(X_test).argmax(dim=-1), 'PyTorch RNN': rnn_model(X_test).argmax(dim=-1)}
print(f'Held-out test set ({len(test)} phrases):')
for name, preds in readings.items():
    token_accuracy = ((preds == Y_idx) & real).sum().item() / real.sum().item()
    exact = ((preds == Y_idx) | ~real).all(dim=1).float().mean().item()
    print(f'  {name}  token accuracy {token_accuracy:.4f}  exact phrases {exact:.4f}')

mistakes = [i for i in range(len(test)) if not ((readings['Scratch RNN'][i] == Y_idx[i]) | ~real[i]).all()]
print(f'\nTest phrases the scratch RNN decoded incorrectly: {len(mistakes)} of {len(test)}')
for i in mistakes[:5]:
    expected, predicted = test.decode(Y_idx[i]), test.decode(readings['Scratch RNN'][i])
    print('  cipher    :', test.decode(X_test[i]))
    print('  expected  :', expected)
    print('  predicted :', predicted)
    print('              ' + ''.join('^' if a != b else ' ' for a, b in zip(expected, predicted)))

x = dataset.encode(letter_text).unsqueeze(0).to(device)
with torch.no_grad():
    scratch_note = dataset.decode(model.forward(x).argmax(dim=-1)[0])
    torch_note = dataset.decode(rnn_model(x).argmax(dim=-1)[0])
exact_note = codecs.encode(letter_text, 'rot_13')
print(f'\nCharacters matching the exact decoding: scratch {sum(a == b for a, b in zip(scratch_note, exact_note))} of {len(exact_note)}, '
      f'PyTorch {sum(a == b for a, b in zip(torch_note, exact_note))} of {len(exact_note)}')
print('\nThe scratch RNN reads the note as:')
print(scratch_note)

### Task 04-1: Reference Output (0 pts.)

Both networks should score at or very near 1.0000 on the test phrases and read all 262 characters of the note correctly.

```text
Held-out test set (150 phrases):
  Scratch RNN  token accuracy 1.0000  exact phrases 1.0000
  PyTorch RNN  token accuracy 1.0000  exact phrases 1.0000

Test phrases the scratch RNN decoded incorrectly: 0 of 150

Characters matching the exact decoding: scratch 262 of 262, PyTorch 262 of 262

The scratch RNN reads the note as:
i loved murdering mr theisenfloyd at the estate the gas was the perfect murder weapon im sure ill get away with it as well hopefully nobody figures out the combination of the padlock play on locker 69 on the second floor of cushing otherwise i am in real trouble
```

### Task 04-2: Written Response (0.125 pts.)

Your test scores are probably perfect, or very close. The two accuracy numbers can still tell very different stories.

1. A 30-character phrase is decoded with one wrong character. What are its token accuracy and its exact-phrase accuracy?
2. Which number would you quote to Caulfield about a locker number, and why?
3. If the network did make mistakes, which characters would you expect them on? (Hint: how often does each digit appear in training, compared with each letter?)

* `[ANSWER]`

### *Story Progression*

The intern runs the note through the trained network and reads the screen aloud, slowly, then much faster:

*"i loved murdering mr theisenfloyd at the estate the gas was the perfect murder weapon..."*

The room goes quiet. The note goes on: whoever wrote it is sure they'll get away with it, as long as nobody figures out the combination of the padlock, **play**, on **locker 69** on the second floor of **Cushing**.

Every character matches the hand-checked decoding.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/story_decoded.png" class="img-responsive" alt="Evidence 3: The intern reads the decoded note aloud while Caulfield and his partner lean in." style="display: block; max-width: 640px; width: 100%; height: auto; margin: 0 auto;"/>
    <figcaption style="text-align: center"><b>Evidence 3: The intern reads the decoded note aloud while Caulfield and his partner lean in.</b></figcaption>
</div>

## Task 05: Report the Results to Caulfield
### Task 05-1: Written Response (0.5 pts.)

1. Report both networks' test token accuracy and exact-sequence accuracy. Did the scratch and PyTorch networks agree? What did the supplied gradient checks establish about your implementation, and what did training establish that they could not?
    * `[ANSWER]`
2. Your decoder reproduced the exact ROT13 decoding of the note. What does that establish, and what does it **not** establish? In particular, does it show who wrote the note, or that its claims are true? What would Caulfield need to check next?
    * `[ANSWER]`
3. ROT13 maps each character independently of its neighbors. Explain why that makes this an easy task for an RNN, and describe a kind of message (or code) for which carrying information back through many time steps would matter much more.
    * `[ANSWER]`

### *Story Progression*

Caulfield reads the decoded note three times, then writes two words on the board: **corroborate it**. A note found in Professor Dingler's desk drawer isn't the same thing as Professor Dingler's confession. Anyone could have printed it, cut it up, and left it there. A network that decodes a message perfectly tells us what the message says. It can't tell us who wrote it, or whether it's true.

But the note makes a claim that can be checked: a padlocked locker on the second floor of Cushing. Before anyone goes looking, Caulfield wants one more opinion. A different kind of network, one that doesn't read left to right and doesn't have to carry a memory through 262 characters, should read the note the same way. That network is built from **attention**, and it's where HW10 begins.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework09/story_corroborate.png" class="img-responsive" alt="Evidence 4: The confession, decoded. Caulfield pins up the next step: corroborate it." style="display: block; max-width: 640px; width: 100%; height: auto; margin: 0 auto;"/>
    <figcaption style="text-align: center"><b>Evidence 4: The confession, decoded. Caulfield pins up the next step: corroborate it.</b></figcaption>
</div>

## Task 06: Export the Completed Notebook (0 pts.)

1. Complete your code and written responses, and run the cells above in order. Compare your results with the expected and reference outputs.
2. **Save your notebook**, including its outputs. Set `GENERATE = True` below when you are ready to export.
3. In Colab, the cell captures the current notebook and its outputs. Locally, it reads the saved notebook at `NOTEBOOK_PATH`; adjust that path if you saved your copy elsewhere.
4. Download **`homework09.html`** and open it in a browser. Check that your name, NETID, code, written answers, and outputs appear.

Submit only **`homework09.html`** on Canvas. The exporter does not rerun your code.

### Task 06-1: Code (0 pts.)

In [ ]:
GENERATE = False
NOTEBOOK_PATH = DATA_ROOT / 'homeworks' / 'homework09' / 'homework09.ipynb'

if GENERATE:
    import json
    import nbformat
    from nbconvert import HTMLExporter
    from pathlib import Path

    try:
        from google.colab import _message, files
    except ImportError:
        current = nbformat.read(NOTEBOOK_PATH, as_version=4)
        html_path = NOTEBOOK_PATH.with_name('homework09.html')
        in_colab = False
    else:
        captured = _message.blocking_request('get_ipynb', timeout_sec=10)
        current = nbformat.reads(json.dumps(captured['ipynb']), as_version=4)
        html_path = Path('/content/homework09.html')
        in_colab = True

    exporter = HTMLExporter()
    exporter.embed_images = True
    report_html, _ = exporter.from_notebook_node(current)
    html_path.write_text(report_html, encoding='utf-8')
    print(f'Exported {html_path.name}. Open it in a browser before submitting.')

    if in_colab:
        files.download(str(html_path))
    else:
        print(f'Saved to: {html_path}')
else:
    print('Export disabled. Set GENERATE = True when ready.')

### Task 06-1: Expected Output (0 pts.)

```text
Export disabled. Set GENERATE = True when ready.
```